In [ ]:
### 지시사항, 입력 문자열로 응답(결과) 문자열 생성하도록 학습하는 방법
  - InstructionDataset class
  - format_input()      : 입력 문자열을 형식에 맞게 변형하는 함수
  - custom_collate_fn() : 데이터 로더에서 데이터 가공하는 함수
    - customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024) : partial 로 custom_collate_fn 적용

In [ ]:
### InstructionDataset 클래스
    - "지시사항 문자와 입력 문자를 입력 문자열로 만들어서 응답 문자를 예측하는 모델 용 데이터셋 클래스

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        ## 1. 텍스트 데이터 토큰화
        self.encoded_texts = []
        for entry in data:
            # 1.1 포맷팅 함수를 이용해 Alpaca 스타일로 지시사항 + 입력 형식의 문자열로 변경
            instruction_plus_input = format_input( entry )

            # 1.2 정답 문구 생성
            response_text = f"\n\n### Response:\n{entry['output']}"

            # 1.3 모델이 이 전체 텍스트(질문 + 답변)을 보고 다음 토큰을 예측하도록 학습됨
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append( tokenizer.encode(full_text) )

    def __len__():
        return len(self.encoded_texts)

    def __getitem__(self, index):
        return self.encoded_texts[index]

In [ ]:
### format_input() : 입력 문자열을 Alpaca 스타일로 만드는 함수

def format_input( entry ) :
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""

    return instruction_text + input_text

In [ ]:
### custom_collate_fn()
    - 데이터 로더에서 배치를 만들 때 사용하는 커스텀 함수
    - 가변 길이의 시퀀스를 배치 최대 길이에 맞춰서 패딩(50256)을 추가
    - 정답(target) 데이터에서 패딩 부분은 손실 계산에서 제외하도록 -100 설정

def custom_collate_fn( batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu" )
    inputs_lst, targets_lst = [], []

    ## 1. 배치 내에서 가장 긴 시퀀스 길이 계산 (패딩을 위해 +1 계산)
    batch_max_length = max( len(item) + 1 for item in batch )

    for item in batch :
        new_item = item.copy()
        
        ## 2. 모든 스퀀스의 문장 끝에 패딩 추가
        new_item += [pad_token_id]

        ## 3. 가장 긴 길이에 맞춰서 패딩 채우기
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        ## 4. inputs, targets 슬라이딩 만들기
        inputs = torch.tensor( padded[ : -1 ] )     # [0 ~ n-1]
        targets = torch.tensor( padded[ 1 : ] )     # [1 ~ n]

        ## 5. target 에서 문장 끝 토큰 제외 패딩 토큰은 마스킹 (loss 계산 시 제외하기 위해)
        mask = targets == pad_token_id
        indices = torch.nozero(mask).sequeeze()     # pad_token_id 인 부분의 index 계산

        if indices.numel() > 1:
            targets[ indices[1]: ] = ignore_index    # indices[0] 부분이 문장 끝 pad_token_id, 그 다음부터 끝까지 -100 채우기

        ## 6. 최대 시쿼스 길이 제한 체크
        if allowed_max_length is not None:
            inputs = inputs[ : allowed_max_length ]
            targets = targets[ : allowed_max_length ]
        
        ## 7. input, target 리스트에 추가
        inputs_lst.append( inputs )
        targets_lst.append( targets )

    ## 8. tensor 로 변환 및 GPU 로 이동
    inputs_tensor = torch.stack( inputs_lst ).to(device)
    targets_tensor = torch.stack( targets_lst ).to(device)

    return inputs_tensor, targets_tensor     



In [ ]:
### 모델 로드 / 지시 튜닝 / 학습 / 평가

## 1. 데이터 로드 및 준비
file_path = "datas/instruction-data.json"
data = download_and_load_file(file_path)

# 1.1 데이터 분할
train_portion = int( len(data) * 0.85 )     # 85%
test_protion = int( len(data) * 0.1 )       # 10%

train_data = data[ : train_portion ]
test_data = data[ train_portion : train_portion + test_protion ]
val_data = data[ train_portion + test_protion : ]

# 1.2 토크나이저 및 device 설정
tokenizer = tiktoken.get_encoding("gpt2")
device = torch.device( "cuda" torch.cuda.is_available() else "cpu" )

# 1.3 custom_collate_fn 함수 설정
customized_collate_fn = partial( custom_collate_fn, device=device, allowed_max_length=1024 )

num_workers = 0
batch_size = 8
torch.manual_seed( 123 )

# 1.4 데이터 셋 및 데이터 로드 생성
train_dataset = InstructionDataset( train_data, tokenizer )
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,                       # shuffle / drop_last
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset( val_data, tokenizer )
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)


## 2. 사전 학습된 모델 로딩
# 실제 학습용 설정: GPT-2 Medium (355M)
BASE_CONFIG = {
    "vocab_size": 50257,     
    "context_length": 1024,  
    "drop_rate": 0.0,        
    "qkv_bias": True         
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

# 로컬에 저장된 모델 가중치 파일 로드
model_name = "gpt2-medium-355M.pth"
model = GPTModel(BASE_CONFIG)
checkpoint = torch.load(model_name, map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint)

model.eval()
model.to(device)


## 3. 모델 미세 조정 (파인 튜닝)
# 3.1 학습 전 초기 손실 확인
with torch.no_grad():
    train_loss = calc_loss_loader( train_loader, model, device, num_batches=5 )
    val_loss = calc_loss_loader( val_loader, model, device, num_batches=5 )

print(f"train loss : {train_loss}" )
print(f"val loss : {val_loss}" )

# 3.2 옵티마이저 설정 및 지시 학습 시작
optimizer = torch.optim.AdamW( model.parameters(), lr=0.00005, weight_decay=0.1 )

num_epochs = 2

start_time = time.time()
train_losses, val_losses, token_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]),
    tokenizer=tokenizer
)
end_time = time.time()
execution_time = (end_time - start_time) / 60


## 4. 학습 결과 모델 저장 및 로드
# 4.1 테스트 데이터 셋 응답 생성
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input( entry )

    # 모델 적용
    token_ids = generate(
        model=model,
        idx=text_to_token_ids( input_text, tokenizer ).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["content_length"],
        eos_id=50256
    )

    generated_text = token_ids_to_text( token_ids, tokenizer )

    # 프롬프트 부분 제거한 순수 응답만 저장
    response_text = generated_text[ len(input_text) : ].replace("### Response:", "").strip()

    test_data[i]["mode_response"] = response_text

## 5. 생성된 응답을 json 으로 저장
test_data_path = "outputs/instruction-data-with-response-standanlone.json"
with open(test_data_path, "w") as f :
    json.dump( test_data, f, indent=4 )


## 6. 모델 가중치 저장 및 재로딩
file_name = "outputs/model.pth"
torch.save(model.state_dict(), file_name)

model = GPTModel(BASE_CONFIG)
checkpoint = torch.load(file_name, map_location=device, weights_only=True)
model.load_state_dict(checkpoint)
model.to(device)